In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ y_{\text{onehot}} $$
$$ y = 0 \rightarrow [1, 0, 0, \dots] $$
$$ y = 1 \rightarrow [0, 1, 0, \dots] $$
$$ y = 2 \rightarrow [0, 0, 1, \dots] $$
$$ \vdots $$
$$ y = C \rightarrow [0, 0, \dots, 1] $$
$$ y_k \in \{0, 1\}, \quad \sum_k y_k = 1 $$
$$ \diamond \diamond \diamond $$

$$ Model = M(W, b) = Wx + b $$
$$ \frac{\partial M}{\partial W} = x $$
$$ \frac{\partial M}{\partial b} = 1 $$
$$ \diamond \diamond \diamond $$

$$ Softmax(M_i) = S_i = \frac{e^{M_i}}{\sum_{k} e^{M_k}} $$
$$
\frac{\partial S_i}{\partial M_j} = S_i (\delta_{ij} - S_j) =
\begin{dcases}
i = j & \frac{\partial S_i}{\partial M_i} = 
\frac{e^{M_i} \sum_{k} e^{M_k} - e^{M_i} e^{M_i}}{\sum_{k} e^{M_k} \sum_{k} e^{M_k}} =
S_i (1 - S_i) \\
\\
i \neq j & \frac{\partial S_i}{\partial M_j} = 
\frac{0 \cdot \sum_{k} e^{M_k} - e^{M_i} e^{M_j}}{\sum_{k} e^{M_k} \sum_{k} e^{M_k}} = 
-S_i S_j  = S_i (0 - S_j)\\
\end{dcases}
$$
$$ \frac{\partial S}{\partial M} = \begin{pmatrix}
S_0 (1 - S_0) & -S_0 S_1 & \cdots & -S_0 S_C \\
-S_1 S_0 & S_1 (1 - S_1) & \cdots & -S_1 S_C \\
\vdots & \vdots & \ddots & \vdots \\
-S_C S_0 & -S_C S_1 & \cdots & S_C (1 - S_C) \\
\end{pmatrix}$$

$$ \diamond \diamond \diamond $$

$$ Loss = L = -\sum_{c \in \mathcal{C}} y_c \log S_c $$

$$ \frac{\partial L}{\partial S} = 

- \sum_{c \in \mathcal{C}} y_c \frac{1}{S_c} = 

- [0_0, \dots, 1_n, \dots, 0_C] \cdot \begin{pmatrix}
\frac{1}{S_0} & 0 & \cdots & 0 \\
0 & \frac{1}{S_1} & \cdots & 0 \\
\vdots & \vdots & \ddots & \vdots \\
0 & 0 & \cdots & \frac{1}{S_C} \\
\end{pmatrix} = 

-y_{onehot} \cdot \frac{1}{S}
$$

$$
\frac{\partial L}{\partial M} = \frac{\partial L}{\partial S} \cdot \frac{\partial S}{\partial M} = 

-y_{onehot}
\cdot 
\begin{pmatrix}
\frac{1}{S_0} & 0 & \cdots & 0 \\
0 & \frac{1}{S_1} & \cdots & 0 \\
\vdots & \vdots & \ddots & \vdots \\
0 & 0 & \cdots & \frac{1}{S_C} \\
\end{pmatrix}
\cdot
\begin{pmatrix}
S_0 (1 - S_0) & -S_0 S_1 & \cdots & -S_0 S_C \\
-S_1 S_0 & S_1 (1 - S_1) & \cdots & -S_1 S_C \\
\vdots & \vdots & \ddots & \vdots \\
-S_C S_0 & -S_C S_1 & \cdots & S_C (1 - S_C) \\
\end{pmatrix} = 

S - y_{onehot}
$$

$$ \frac{\partial L}{\partial W} = \frac{\partial L}{\partial M} \cdot \frac{\partial M}{\partial W} = (S - y_{onehot}) \cdot x $$

$$ \frac{\partial L}{\partial b} = \frac{\partial L}{\partial M} \cdot \frac{\partial M}{\partial b} = (S - y_{onehot}) \cdot 1 $$

$$ \diamond \diamond \diamond $$


In [ ]:
from torch import float32, randn, tensor
from torch.nn import Linear
from torch.nn.functional import softmax, cross_entropy, one_hot
from torch.optim import SGD


import import_ipynb
from common import assert_eq, assert_ge, check_eq, Patient, T # type: ignore


def logistic_regression_nc_sgd_gradient(X, y, epochs=2000, lr=0.1) -> tuple[float, callable]:
    """
    Perform logistic regression using Stochastic Gradient Descent (SGD) with autograd.

    Args:
        X: Input features of shape (Samples, Features)
        y: Target values of shape (Samples, 1)
        epochs: Number of training epochs
        lr: Learning rate

    Returns:
        A tuple containing the final loss and a prediction function that takes new input data and returns predicted probabilities.
    """

    ((s, f), c) = (X.shape, len(set(y.squeeze().tolist())))

    assert_eq(X.shape, (s, f))
    assert_eq(y.shape, (s, 1))

    y = one_hot(y.squeeze().long(), num_classes=c).float()
    assert_eq(y.shape, (s, c))

    W = randn(f, c) * 0.1
    assert_eq(W.shape, (f, c))
    
    b = randn(c) * 0.1
    assert_eq(b.shape, (c,))

    for _ in range(epochs):
        logits = X @ W + b
        assert_eq(logits.shape, (s, c))
        
        probs = softmax(logits, dim=1)
        assert_eq(probs.shape, (s, c))

        dL_dz = (probs - y) / s
        assert_eq(dL_dz.shape, (s, c))

        dL_dW = X.T @ dL_dz
        assert_eq(dL_dW.shape, (f, c))
        
        dL_db = dL_dz.sum(dim=0)
        assert_eq(dL_db.shape, (c,))

        W -= lr * dL_dW
        b -= lr * dL_db

        #
        # In the autograd version, computing the loss is required because it serves 
        # as the root of the computational graph for backpropagation. 
        #
        # In the manual‑gradient version, the loss value is not needed for the weight update itself,
        # it is computed only to monitor training progress.
        #
        loss = -(y * (probs + 1e-9).log()).sum(dim=1).mean()
    
    return (loss.item(), lambda x: softmax(x @ W + b, dim=0))


def _test_logistic_regression_nc_sgd_gradient(epochs=2000, lr=0.1) -> None:
    """
    Test the logistic regression implementation on a synthetic dataset of patients. 
    The dataset consists of three classes: healthy (0), viral (1), and bacterial (2). 
    The model is trained on 70 samples and then evaluated on 30 samples (10 per class). 
    The test checks if the model achieves at least 90% accuracy on the evaluation set.
    """

    training_data = T([Patient([0.5, 0.25, 0.25]).data for _ in range(70)])

    X = training_data[:, :-1]
    X[:, 0] /= 100 # Temperature scaling
    X[:, 5] /= 200 # CRP scaling   

    y = training_data[:, -1].unsqueeze(1)

    (_, model) = logistic_regression_nc_sgd_gradient(X, y, epochs, lr)

    total = 0
    correct = 0

    for d in ([Patient([1.0, 0.0, 0.0]).data for _ in range(10)] + 
              [Patient([0.0, 1.0, 0.0]).data for _ in range(10)] + 
              [Patient([0.0, 0.0, 1.0]).data for _ in range(10)]):
        x = T(d[:-1])
        y = T(d[-1])

        x[0] /= 100
        x[5] /= 200
        
        total += 1
        correct += check_eq(softmax(model(x), dim=0).argmax(), y)

    assert_ge(correct / total, 0.9)


if __name__ == "__main__":
    _test_logistic_regression_nc_sgd_gradient()